# Baseline — Fashion-IQ: Composed Image Retrieval

**Competition:** given a **reference image** of a clothing item and a **modifier text**
("is more casual with shorter sleeves"), find which of the 10 **candidate images** is the
target the text describes. Based on the FashionIQ dataset.

- **Task:** pick the correct candidate (0–9) for each query
- **Metric:** accuracy
- **Kaggle link:** _TODO: add link_

**Approach (zero-shot CLIP):** CLIP embeds images and text into the *same* vector
space. A classic composed-retrieval baseline scores each candidate by cosine
similarity to the **sum of the reference-image embedding and the modifier-text
embedding** — no training at all.

In [1]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval().to(device)
proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

(2000, 16) (500, 14)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [2]:
# Embed every unique image once
cand_cols = [f"candidate_{i}" for i in range(10)]
img_ids = pd.unique(pd.concat([train["reference_image"], test["reference_image"],
                               train[cand_cols].stack(), test[cand_cols].stack()]))
print(len(img_ids), "unique images")

@torch.no_grad()
def embed_images(ids, batch_size=64):
    out = {}
    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i+batch_size]
        ims = [Image.open(f"{DATA_DIR}/images/{x}.jpg").convert("RGB") for x in batch_ids]
        inp = proc(images=ims, return_tensors="pt").to(device)
        emb = model.visual_projection(model.vision_model(**inp).pooler_output)
        emb = torch.nn.functional.normalize(emb, dim=-1).cpu().numpy()
        out.update(dict(zip(batch_ids, emb)))
    return out

img_emb = embed_images(img_ids)

@torch.no_grad()
def embed_texts(texts, batch_size=128):
    embs = []
    for i in range(0, len(texts), batch_size):
        inp = proc(text=list(texts[i:i+batch_size]), return_tensors="pt",
                   padding=True, truncation=True).to(device)
        emb = model.text_projection(model.text_model(**inp).pooler_output)
        embs.append(torch.nn.functional.normalize(emb, dim=-1).cpu().numpy())
    return np.vstack(embs)

4791 unique images


In [3]:
def predict(df):
    txt = embed_texts(df["modifier_text"].values)
    preds = np.zeros(len(df), dtype=int)
    for i, (_, row) in enumerate(df.iterrows()):
        query = img_emb[row["reference_image"]] + txt[i]          # compose image + text
        query /= np.linalg.norm(query)
        cands = np.stack([img_emb[row[c]] for c in cand_cols])
        preds[i] = int(np.argmax(cands @ query))
    return preds

train_pred = predict(train)
acc = (train_pred == train["target"].values).mean()
print(f"Zero-shot accuracy on train queries: {acc:.4f}  (random guess = 0.10)")

Zero-shot accuracy on train queries: 0.5865  (random guess = 0.10)


In [4]:
sub = pd.DataFrame({"query_id": test["query_id"], "target": predict(test)})
sub.to_csv("submission.csv", index=False)
sub.head()

,query_id,target
0,q_000000,7
1,q_000001,9
2,q_000002,6
3,q_000003,3
4,q_000004,6


## Ideas to improve

- Weight the composition: `img + w * text`, tune `w` on the train queries.
- Use a larger CLIP (ViT-L/14) or a fashion-tuned CLIP (e.g. FashionCLIP).
- Train a small **combiner network** on the train queries (Combiner / Pic2Word style).
- Add the category ("shirt", "dress", "toptee") to the text prompt.
